# Baseline de Regresión Logística — Predicción de `mora`

Este notebook construye un **modelo baseline multiclase de regresión logística** utilizando las bases limpias de:

- `train_mora_limpio.csv`
- `validation_mora_limpio.csv`
- `test_mora_limpio.csv`

## Criterios principales

El modelo utiliza `train` para aprender los parámetros, `validation` para evaluar durante el desarrollo y `test` como evaluación final.

Se excluyen del modelo:

- `ID`: es un identificador y no una variable predictiva generalizable.
- `fecha_reporte`: se excluye del baseline para evitar introducir una dimensión temporal sin un diseño específico de validación temporal.
- `mora`: es la variable objetivo.
- `atraso`: se excluye porque sus valores están directamente relacionados con las categorías de mora y, por tanto, producirían fuga de información.

El preprocesamiento estandariza las variables numéricas y codifica las variables categóricas con One-Hot Encoding.

## Gráficas incluidas

1. Distribución de `mora`.
2. Distribución de nulos antes del modelado.
3. F1-macro de validación cruzada.
4. Comparación de métricas entre validation y test.
5. Matriz de confusión absoluta de validation.
6. Matriz de confusión normalizada de validation.
7. Matriz de confusión absoluta de test.
8. Matriz de confusión normalizada de test.
9. Curvas ROC One-vs-Rest.
10. Curvas Precision-Recall One-vs-Rest.
11. Importancia aproximada de variables mediante coeficientes de la regresión logística.

## Métricas

Se reportan:

- Accuracy.
- Balanced Accuracy.
- Precision macro.
- Recall macro.
- F1 macro.
- F1 weighted.
- Log Loss.
- ROC-AUC macro One-vs-Rest.
- Average Precision macro.

El **F1-macro** es especialmente útil aquí porque las clases de mora no tienen la misma frecuencia.


In [ ]:
# Importamos sys para revisar la versión de Python.
import sys
# Mostramos la versión de Python que está ejecutando Colab.
print("Python:", sys.version)
# Mostramos un mensaje indicando que las bibliotecas necesarias suelen estar preinstaladas en Colab.
print("Google Colab normalmente ya incluye pandas, numpy, matplotlib y scikit-learn.")

In [ ]:
# Importamos la función de carga de archivos de Google Colab.
from google.colab import files
# Abrimos el selector para cargar las tres bases limpias.
archivos_subidos = files.upload()
# Mostramos los nombres de todos los archivos cargados.
print("Archivos cargados:", list(archivos_subidos.keys()))

In [ ]:
# Importamos Path para manejar rutas de archivos.
from pathlib import Path
# Importamos pandas para manipular las bases de datos.
import pandas as pd
# Importamos numpy para cálculos numéricos.
import numpy as np
# Importamos matplotlib para generar las gráficas.
import matplotlib.pyplot as plt
# Importamos métricas de clasificación de scikit-learn.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    log_loss,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
# Importamos el transformador para procesar variables numéricas y categóricas.
from sklearn.compose import ColumnTransformer
# Importamos el escalador de variables numéricas.
from sklearn.preprocessing import StandardScaler
# Importamos One-Hot Encoding para variables categóricas.
from sklearn.preprocessing import OneHotEncoder
# Importamos Pipeline para encadenar preprocesamiento y modelo.
from sklearn.pipeline import Pipeline
# Importamos la regresión logística.
from sklearn.linear_model import LogisticRegression
# Importamos validación cruzada estratificada.
from sklearn.model_selection import StratifiedKFold, cross_val_score
# Importamos joblib para guardar el modelo entrenado.
import joblib

# Definimos la carpeta donde Colab coloca los archivos subidos.
BASE = Path("/content")

In [ ]:
# Definimos una función que localiza un archivo por una parte de su nombre.
def localizar_archivo(fragmento):
    # Buscamos todos los CSV de la carpeta de trabajo.
    candidatos = list(BASE.glob("*.csv"))
    # Recorremos cada archivo encontrado.
    for archivo in candidatos:
        # Convertimos el nombre a minúsculas para comparar sin importar el caso.
        nombre = archivo.name.lower()
        # Comprobamos si contiene el fragmento solicitado.
        if fragmento.lower() in nombre:
            # Devolvemos la primera coincidencia válida.
            return archivo
    # Generamos un error claro si no encontramos el archivo.
    raise FileNotFoundError(f"No se encontró un CSV que contenga: {fragmento}")

# Localizamos la base limpia de train.
RUTA_TRAIN = localizar_archivo("train_mora_limpio")
# Localizamos la base limpia de validation.
RUTA_VALIDATION = localizar_archivo("validation_mora_limpio")
# Localizamos la base limpia de test.
RUTA_TEST = localizar_archivo("test_mora_limpio")

# Mostramos las rutas encontradas.
print("Train:", RUTA_TRAIN)
print("Validation:", RUTA_VALIDATION)
print("Test:", RUTA_TEST)

In [ ]:
# Leemos la base limpia de train.
train = pd.read_csv(RUTA_TRAIN, encoding="utf-8-sig")
# Leemos la base limpia de validation.
validation = pd.read_csv(RUTA_VALIDATION, encoding="utf-8-sig")
# Leemos la base limpia de test.
test = pd.read_csv(RUTA_TEST, encoding="utf-8-sig")

# Mostramos las dimensiones de cada partición.
print("Dimensiones train:", train.shape)
print("Dimensiones validation:", validation.shape)
print("Dimensiones test:", test.shape)

# Calculamos el total de valores nulos de train.
nulos_train = int(train.isna().sum().sum())
# Calculamos el total de valores nulos de validation.
nulos_validation = int(validation.isna().sum().sum())
# Calculamos el total de valores nulos de test.
nulos_test = int(test.isna().sum().sum())

# Mostramos el total de nulos de cada partición.
print("Nulos train:", nulos_train)
print("Nulos validation:", nulos_validation)
print("Nulos test:", nulos_test)

# Mostramos una muestra de la base para confirmar la estructura.
display(train.head())

In [ ]:
# Calculamos la frecuencia de cada categoría de mora.
frecuencias_mora = train["mora"].value_counts()
# Calculamos el porcentaje de cada categoría de mora.
porcentajes_mora = train["mora"].value_counts(normalize=True).mul(100)

# Creamos una figura independiente para la distribución de la variable objetivo.
plt.figure(figsize=(12, 6))
# Creamos las barras con las frecuencias observadas.
plt.bar(frecuencias_mora.index, frecuencias_mora.values)
# Agregamos las cantidades encima de las barras.
for posicion, valor in enumerate(frecuencias_mora.values):
    # Escribimos el número exacto de observaciones.
    plt.text(posicion, valor + max(frecuencias_mora.values) * 0.01, f"{valor:,}", ha="center")
# Definimos el título del gráfico.
plt.title("Distribución de la variable objetivo: mora")
# Definimos el nombre del eje X.
plt.xlabel("Categoría de mora")
# Definimos el nombre del eje Y.
plt.ylabel("Número de observaciones")
# Rotamos las etiquetas para mejorar la lectura.
plt.xticks(rotation=35, ha="right")
# Ajustamos automáticamente el espacio de la figura.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

# Mostramos la tabla de distribución con frecuencia y porcentaje.
tabla_mora = pd.DataFrame({
    "Frecuencia": frecuencias_mora,
    "Porcentaje": porcentajes_mora.round(2),
})
# Ordenamos la tabla según la frecuencia observada.
tabla_mora = tabla_mora.sort_values("Frecuencia", ascending=False)
# Mostramos la tabla.
display(tabla_mora)

In [ ]:
# Calculamos el porcentaje de nulos por columna en train.
nulos_por_columna = train.isna().mean().mul(100).sort_values(ascending=False)
# Filtramos las columnas que realmente tienen nulos.
nulos_por_columna = nulos_por_columna[nulos_por_columna > 0]

# Creamos una figura independiente para mostrar los nulos.
plt.figure(figsize=(12, 5))
# Dibujamos las barras de porcentaje de nulos.
plt.bar(nulos_por_columna.index, nulos_por_columna.values)
# Definimos el título.
plt.title("Valores faltantes por variable en la base limpia de train")
# Definimos el eje X.
plt.xlabel("Variable")
# Definimos el eje Y.
plt.ylabel("Porcentaje de nulos")
# Rotamos las etiquetas.
plt.xticks(rotation=90)
# Ajustamos los márgenes.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

# Indicamos explícitamente si no existen valores faltantes.
if nulos_por_columna.empty:
    # Informamos que la base ya no contiene nulos.
    print("La base limpia no contiene valores faltantes.")

In [ ]:
# Definimos las columnas que no entrarán en el modelo.
EXCLUIR = ["ID", "fecha_reporte", "mora", "atraso"]

# Creamos la lista de predictores conservando el esquema de la base.
columnas_predictoras = [columna for columna in train.columns if columna not in EXCLUIR]

# Identificamos las variables numéricas entre los predictores.
columnas_numericas = [
    columna
    for columna in columnas_predictoras
    if pd.api.types.is_numeric_dtype(train[columna])
]

# Identificamos las variables categóricas entre los predictores.
columnas_categoricas = [
    columna
    for columna in columnas_predictoras
    if train[columna].dtype == "object"
]

# Definimos X para train.
X_train = train[columnas_predictoras]
# Definimos y para train.
y_train = train["mora"]

# Definimos X para validation.
X_validation = validation[columnas_predictoras]
# Definimos y para validation.
y_validation = validation["mora"]

# Definimos X para test.
X_test = test[columnas_predictoras]
# Definimos y para test.
y_test = test["mora"]

# Mostramos las variables excluidas.
print("Variables excluidas:", EXCLUIR)
# Mostramos las variables numéricas.
print("Variables numéricas:", columnas_numericas)
# Mostramos las variables categóricas.
print("Variables categóricas:", columnas_categoricas)
# Mostramos el total de predictores antes de codificar.
print("Total de predictores antes de One-Hot Encoding:", len(columnas_predictoras))

In [ ]:
# Creamos el transformador para las variables numéricas.
transformador_numerico = StandardScaler()

# Creamos el transformador para las variables categóricas.
# El parámetro handle_unknown evita errores si aparece una categoría nueva en validation o test.
try:
    # Usamos la versión moderna de scikit-learn.
    transformador_categorico = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    # Usamos esta alternativa para versiones antiguas de scikit-learn.
    transformador_categorico = OneHotEncoder(handle_unknown="ignore", sparse=True)

# Construimos el preprocesador con las transformaciones correspondientes.
preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", transformador_numerico, columnas_numericas),
        ("categoricas", transformador_categorico, columnas_categoricas),
    ]
)

# Creamos la regresión logística multinomial.
# Se deja class_weight=None porque este experimento representa el baseline sin ponderación de clases.
modelo_logistico = LogisticRegression(
    max_iter=5000,
    solver="lbfgs",
    class_weight=None,
)

# Encadenamos el preprocesamiento y la regresión logística en un único pipeline.
pipeline_logistico = Pipeline(
    steps=[
        ("preprocesamiento", preprocesador),
        ("modelo", modelo_logistico),
    ]
)

# Entrenamos el pipeline exclusivamente con train.
pipeline_logistico.fit(X_train, y_train)

# Informamos que el entrenamiento terminó correctamente.
print("Modelo baseline entrenado correctamente.")

In [ ]:
# Definimos una validación cruzada estratificada de cinco particiones.
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Evaluamos F1-macro dentro de train para revisar estabilidad del baseline.
scores_cv = cross_val_score(
    pipeline_logistico,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
)

# Mostramos los cinco resultados de F1-macro.
print("F1-macro por fold:", np.round(scores_cv, 4))
# Calculamos el promedio de F1-macro.
promedio_cv = scores_cv.mean()
# Calculamos la desviación estándar de F1-macro.
desviacion_cv = scores_cv.std()

# Mostramos el resumen de validación cruzada.
print(f"F1-macro medio: {promedio_cv:.4f}")
print(f"Desviación estándar: {desviacion_cv:.4f}")

# Creamos una figura independiente para los resultados de validación cruzada.
plt.figure(figsize=(8, 5))
# Dibujamos los puntos correspondientes a cada fold.
plt.plot(range(1, len(scores_cv) + 1), scores_cv, marker="o")
# Dibujamos una línea horizontal con el promedio.
plt.axhline(promedio_cv, linestyle="--")
# Definimos el título.
plt.title("Validación cruzada de F1-macro en train")
# Definimos el eje X.
plt.xlabel("Fold")
# Definimos el eje Y.
plt.ylabel("F1-macro")
# Fijamos los folds en el eje X.
plt.xticks(range(1, len(scores_cv) + 1))
# Ajustamos el diseño.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

In [ ]:
# Definimos una función para calcular todas las métricas principales.
def evaluar_modelo(modelo, X, y, nombre):
    # Generamos las clases predichas.
    predicciones = modelo.predict(X)
    # Generamos las probabilidades de cada clase.
    probabilidades = modelo.predict_proba(X)
    # Recuperamos el orden oficial de las clases.
    clases = modelo.named_steps["modelo"].classes_

    # Calculamos Accuracy.
    accuracy = accuracy_score(y, predicciones)
    # Calculamos Balanced Accuracy.
    balanced_accuracy = balanced_accuracy_score(y, predicciones)
    # Calculamos Precision macro.
    precision_macro = precision_score(y, predicciones, average="macro", zero_division=0)
    # Calculamos Recall macro.
    recall_macro = recall_score(y, predicciones, average="macro", zero_division=0)
    # Calculamos F1 macro.
    f1_macro = f1_score(y, predicciones, average="macro", zero_division=0)
    # Calculamos F1 weighted.
    f1_weighted = f1_score(y, predicciones, average="weighted", zero_division=0)
    # Calculamos Log Loss usando las probabilidades.
    perdida_logaritmica = log_loss(y, probabilidades, labels=clases)
    # Convertimos la variable objetivo a formato binario One-vs-Rest.
    y_binario = np.column_stack([(y == clase).astype(int) for clase in clases])
    # Calculamos ROC-AUC macro con estrategia One-vs-Rest.
    roc_auc_macro = roc_auc_score(
        y_binario,
        probabilidades,
        average="macro",
    )
    # Calculamos Average Precision macro.
    average_precision_macro = average_precision_score(
        y_binario,
        probabilidades,
        average="macro",
    )

    # Construimos un diccionario con todas las métricas.
    resultados = {
        "Partición": nombre,
        "Accuracy": accuracy,
        "Balanced Accuracy": balanced_accuracy,
        "Precision Macro": precision_macro,
        "Recall Macro": recall_macro,
        "F1 Macro": f1_macro,
        "F1 Weighted": f1_weighted,
        "Log Loss": perdida_logaritmica,
        "ROC-AUC Macro OvR": roc_auc_macro,
        "Average Precision Macro": average_precision_macro,
    }

    # Generamos el reporte detallado por clase.
    reporte = pd.DataFrame(
        classification_report(
            y,
            predicciones,
            output_dict=True,
            zero_division=0,
        )
    ).T

    # Devolvemos el resultado, las predicciones, las probabilidades y el reporte.
    return resultados, predicciones, probabilidades, reporte

In [ ]:
# Evaluamos el modelo sobre validation.
resultados_validation, pred_validation, proba_validation, reporte_validation = evaluar_modelo(
    pipeline_logistico,
    X_validation,
    y_validation,
    "Validation",
)

# Evaluamos el modelo sobre test.
resultados_test, pred_test, proba_test, reporte_test = evaluar_modelo(
    pipeline_logistico,
    X_test,
    y_test,
    "Test",
)

# Construimos una tabla resumen de validation y test.
tabla_metricas = pd.DataFrame([
    resultados_validation,
    resultados_test,
])

# Redondeamos las métricas para facilitar la lectura.
tabla_metricas_redondeada = tabla_metricas.round(4)

# Mostramos la tabla de métricas.
display(tabla_metricas_redondeada)

# Mostramos el reporte por clase para validation.
print("Reporte de clasificación — Validation")
display(reporte_validation.round(4))

# Mostramos el reporte por clase para test.
print("Reporte de clasificación — Test")
display(reporte_test.round(4))

In [ ]:
# Seleccionamos las métricas que se compararán visualmente.
metricas_grafica = [
    "Accuracy",
    "Balanced Accuracy",
    "Precision Macro",
    "Recall Macro",
    "F1 Macro",
    "F1 Weighted",
    "ROC-AUC Macro OvR",
    "Average Precision Macro",
]

# Definimos las posiciones de las métricas.
posiciones = np.arange(len(metricas_grafica))
# Definimos el ancho de cada barra.
ancho = 0.35

# Creamos una figura independiente para comparar validation y test.
plt.figure(figsize=(14, 6))
# Dibujamos las barras de validation.
plt.bar(
    posiciones - ancho / 2,
    tabla_metricas.loc[0, metricas_grafica].values,
    ancho,
    label="Validation",
)
# Dibujamos las barras de test.
plt.bar(
    posiciones + ancho / 2,
    tabla_metricas.loc[1, metricas_grafica].values,
    ancho,
    label="Test",
)
# Definimos el título.
plt.title("Comparación de métricas entre Validation y Test")
# Definimos el eje X.
plt.xlabel("Métrica")
# Definimos el eje Y.
plt.ylabel("Valor")
# Colocamos las etiquetas de las métricas.
plt.xticks(posiciones, metricas_grafica, rotation=35, ha="right")
# Mostramos la leyenda.
plt.legend()
# Ajustamos el espacio.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

In [ ]:
# Definimos una función para dibujar una matriz de confusión.
def graficar_matriz_confusion(y_real, y_predicho, clases, titulo, normalizada=False):
    # Calculamos la matriz de confusión.
    matriz = confusion_matrix(y_real, y_predicho, labels=clases)

    # Normalizamos por fila cuando se solicita.
    if normalizada:
        # Calculamos el total de cada clase real.
        totales = matriz.sum(axis=1, keepdims=True)
        # Evitamos divisiones entre cero.
        totales[totales == 0] = 1
        # Convertimos los conteos a proporciones.
        matriz = matriz / totales

    # Creamos una figura independiente.
    plt.figure(figsize=(10, 8))
    # Mostramos la matriz como imagen.
    imagen = plt.imshow(matriz, aspect="auto")
    # Agregamos la barra lateral con la escala.
    plt.colorbar(imagen)
    # Definimos el título.
    plt.title(titulo)
    # Definimos el eje X.
    plt.xlabel("Clase predicha")
    # Definimos el eje Y.
    plt.ylabel("Clase real")
    # Definimos las marcas del eje X.
    plt.xticks(np.arange(len(clases)), clases, rotation=35, ha="right")
    # Definimos las marcas del eje Y.
    plt.yticks(np.arange(len(clases)), clases)

    # Recorremos todas las celdas de la matriz para mostrar su valor.
    for fila in range(matriz.shape[0]):
        # Recorremos las columnas de cada fila.
        for columna in range(matriz.shape[1]):
            # Elegimos el formato según sea matriz normalizada o absoluta.
            texto = f"{matriz[fila, columna]:.2f}" if normalizada else f"{int(matriz[fila, columna])}"
            # Escribimos el valor dentro de la celda.
            plt.text(columna, fila, texto, ha="center", va="center")

    # Ajustamos el diseño.
    plt.tight_layout()
    # Mostramos la gráfica.
    plt.show()

# Recuperamos el orden de las clases utilizado por el modelo.
CLASES = pipeline_logistico.named_steps["modelo"].classes_

# Graficamos la matriz absoluta de validation.
graficar_matriz_confusion(
    y_validation,
    pred_validation,
    CLASES,
    "Matriz de confusión — Validation",
    normalizada=False,
)

# Graficamos la matriz normalizada de validation.
graficar_matriz_confusion(
    y_validation,
    pred_validation,
    CLASES,
    "Matriz de confusión normalizada — Validation",
    normalizada=True,
)

# Graficamos la matriz absoluta de test.
graficar_matriz_confusion(
    y_test,
    pred_test,
    CLASES,
    "Matriz de confusión — Test",
    normalizada=False,
)

# Graficamos la matriz normalizada de test.
graficar_matriz_confusion(
    y_test,
    pred_test,
    CLASES,
    "Matriz de confusión normalizada — Test",
    normalizada=True,
)

In [ ]:
# Convertimos las etiquetas de validation a una matriz binaria One-vs-Rest.
y_validation_bin = np.column_stack([
    (y_validation == clase).astype(int)
    for clase in CLASES
])

# Creamos una figura independiente para las curvas ROC.
plt.figure(figsize=(11, 8))

# Recorremos cada clase para calcular su curva ROC.
for indice, clase in enumerate(CLASES):
    # Calculamos la tasa de falsos positivos y verdaderos positivos.
    fpr, tpr, _ = roc_curve(y_validation_bin[:, indice], proba_validation[:, indice])
    # Calculamos el área bajo la curva de esa clase.
    auc_clase = auc(fpr, tpr)
    # Dibujamos la curva de la clase.
    plt.plot(fpr, tpr, label=f"{clase} — AUC={auc_clase:.3f}")

# Dibujamos la línea de referencia de un clasificador aleatorio.
plt.plot([0, 1], [0, 1], linestyle="--")
# Definimos el título.
plt.title("Curvas ROC One-vs-Rest — Validation")
# Definimos el eje X.
plt.xlabel("Tasa de falsos positivos")
# Definimos el eje Y.
plt.ylabel("Tasa de verdaderos positivos")
# Mostramos la leyenda.
plt.legend(loc="lower right", fontsize=8)
# Ajustamos el espacio.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

# Calculamos el ROC-AUC macro de validation.
roc_auc_validation = roc_auc_score(
    y_validation_bin,
    proba_validation,
    average="macro",
)

# Mostramos el ROC-AUC macro.
print(f"ROC-AUC macro OvR en validation: {roc_auc_validation:.4f}")

In [ ]:
# Creamos una figura independiente para las curvas Precision-Recall.
plt.figure(figsize=(11, 8))

# Recorremos cada clase para calcular Precision-Recall.
for indice, clase in enumerate(CLASES):
    # Calculamos precisión, recall y umbrales para cada clase.
    precision, recall, _ = precision_recall_curve(
        y_validation_bin[:, indice],
        proba_validation[:, indice],
    )
    # Calculamos Average Precision de la clase.
    ap_clase = average_precision_score(
        y_validation_bin[:, indice],
        proba_validation[:, indice],
    )
    # Dibujamos la curva Precision-Recall.
    plt.plot(
        recall,
        precision,
        label=f"{clase} — AP={ap_clase:.3f}",
    )

# Definimos el título.
plt.title("Curvas Precision-Recall One-vs-Rest — Validation")
# Definimos el eje X.
plt.xlabel("Recall")
# Definimos el eje Y.
plt.ylabel("Precision")
# Mostramos la leyenda.
plt.legend(loc="lower left", fontsize=8)
# Ajustamos el espacio.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

# Calculamos Average Precision macro.
ap_macro_validation = average_precision_score(
    y_validation_bin,
    proba_validation,
    average="macro",
)

# Mostramos Average Precision macro.
print(f"Average Precision macro en validation: {ap_macro_validation:.4f}")

In [ ]:
# Extraemos los nombres de las variables después del escalado y One-Hot Encoding.
nombres_variables_transformadas = pipeline_logistico.named_steps[
    "preprocesamiento"
].get_feature_names_out()

# Extraemos los coeficientes del modelo logístico.
coeficientes = pipeline_logistico.named_steps["modelo"].coef_

# Calculamos, para cada variable transformada, el mayor valor absoluto de coeficiente entre clases.
impacto_maximo = np.max(np.abs(coeficientes), axis=0)

# Ordenamos las variables por impacto absoluto descendente.
indices_top = np.argsort(impacto_maximo)[-20:][::-1]

# Seleccionamos los nombres de las 20 variables más influyentes.
top_nombres = nombres_variables_transformadas[indices_top]
# Seleccionamos los impactos asociados.
top_impactos = impacto_maximo[indices_top]

# Creamos una figura independiente.
plt.figure(figsize=(12, 8))
# Dibujamos las barras horizontales.
plt.barh(top_nombres[::-1], top_impactos[::-1])
# Definimos el título.
plt.title("20 variables con mayor |coeficiente| entre las clases")
# Definimos el eje X.
plt.xlabel("Máximo valor absoluto del coeficiente")
# Definimos el eje Y.
plt.ylabel("Variable transformada")
# Ajustamos el diseño.
plt.tight_layout()
# Mostramos la gráfica.
plt.show()

# Creamos una tabla con las variables más influyentes.
tabla_coeficientes = pd.DataFrame({
    "Variable transformada": top_nombres,
    "Máximo |coeficiente|": top_impactos,
})

# Mostramos la tabla redondeada.
display(tabla_coeficientes.round(4))

In [ ]:
# Definimos la ruta donde guardaremos el modelo completo.
RUTA_MODELO = BASE / "baseline_regresion_logistica_mora.joblib"
# Guardamos el pipeline completo para conservar preprocesamiento y modelo.
joblib.dump(pipeline_logistico, RUTA_MODELO)

# Guardamos las métricas en un CSV para documentar el experimento.
RUTA_METRICAS = BASE / "metricas_baseline_logistica_mora.csv"
# Exportamos la tabla de métricas.
tabla_metricas_redondeada.to_csv(
    RUTA_METRICAS,
    index=False,
    encoding="utf-8-sig",
)

# Mostramos la ubicación de los archivos.
print("Modelo guardado en:", RUTA_MODELO)
print("Métricas guardadas en:", RUTA_METRICAS)

In [ ]:
# Importamos la función de descarga de Colab.
from google.colab import files

# Descomenta las siguientes líneas para descargar el modelo.
# files.download(str(RUTA_MODELO))

# Descomenta la siguiente línea para descargar las métricas.
# files.download(str(RUTA_METRICAS))

## Lectura recomendada de los resultados

Para presentar el baseline, conviene reportar primero **F1-macro**, **Balanced Accuracy** y **ROC-AUC macro**, acompañados por la matriz de confusión.

La comparación entre validation y test permite comprobar si el comportamiento del baseline se mantiene sobre datos no utilizados durante el ajuste.

Las curvas ROC muestran la capacidad de discriminación One-vs-Rest de cada categoría, mientras que Precision-Recall permite observar el comportamiento de precisión y recall de cada clase.

La gráfica de coeficientes debe interpretarse como **asociación dentro del modelo**, no como causalidad. El signo y magnitud de un coeficiente dependen también del escalado, de la codificación categórica y de la presencia de otras variables correlacionadas.

### Importante

Este baseline excluye `atraso` para evitar fuga de información respecto de `mora`. Esto hace que las métricas sean más representativas de un escenario donde el objetivo es estimar la categoría de mora a partir del resto de características disponibles.
